# Vector Database SDK - Quick Start Guide

A simple guide to using the Vector Database Python SDK.

**Requirements:**
- API server running on `http://localhost:8000`
- Cohere API key in `.env` file

## Setup

In [51]:
import os
from dotenv import load_dotenv
import cohere
from vector_db.sdk import VectorDBClient

load_dotenv()

client = VectorDBClient(base_url=os.getenv("API_BASE_URL", "http://localhost:8000/api/v1"))
co = cohere.Client(os.getenv("COHERE_API_KEY"))

## Create a Library

In [52]:
library = client.create_library(
    name="My First Library",
    index_type="hnsw",
    distance_metric="cosine"
)

## Create a Document

In [53]:
document = client.create_document(
    library_id=library.id,
    name="Sample Document"
)

## Prepare Some Text Data

In [54]:
texts = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is a subset of artificial intelligence",
    "Python is a popular programming language",
    "Vector databases enable semantic search",
    "Natural language processing helps computers understand text"
]

## Generate Embeddings

In [55]:
response = co.embed(
    texts=texts,
    model="embed-english-v3.0",
    input_type="search_document"
)
embeddings = response.embeddings

## Add Chunks

In [56]:
for text, embedding in zip(texts, embeddings):
    client.create_chunk(
        document_id=document.id,
        text=text,
        embedding=embedding
    )

## Search

In [57]:
query = "What is AI?"

query_embedding = co.embed(
    texts=[query],
    model="embed-english-v3.0",
    input_type="search_query"
).embeddings[0]

results = client.search_library(
    library_id=library.id,
    query=query_embedding,
    top_k=3
)

for result in results.results:
    print(f"Score: {result.score:.4f}")
    print(f"Text: {result.chunk.text}")
    print()

Score: 0.6615
Text: Machine learning is a subset of artificial intelligence

Score: 0.5946
Text: Natural language processing helps computers understand text

Score: 0.5623
Text: Python is a popular programming language



## Add Metadata

In [75]:
new_chunk = client.create_chunk(
    document_id=document.id,
    text="Databases are a key component in information systems",
    embedding=co.embed(
        texts=["Databases store and organize data efficiently"],
        model="embed-english-v3.0",
        input_type="search_document"
    ).embeddings[0],
    metadata={
        "category": "database",
        "difficulty": "beginner"
    }
)

## Search with Filters

In [77]:
results = client.search_library(
    library_id=library.id,
    query=query_embedding,
    top_k=5,
    filters={"category": "database"}
)

for result in results.results:
    print(f"{result.chunk.text}")

Databases store and organize data efficiently
Databases store and organize data efficiently
Databases store and organize data efficiently
Databases are a key component in information systems
Databases are a key component in information systems


## List Libraries

In [ ]:
libraries = client.list_libraries()

for lib in libraries.items:
    print(f"{lib.name} - {lib.index_config.index_type}")

## List Documents

In [ ]:
documents = client.list_documents(library_id=library.id)

for doc in documents.items:
    print(doc.name)

## List Chunks

In [ ]:
chunks = client.list_chunks(document_id=document.id)

for chunk in chunks.items:
    print(chunk.text)

## Get a Specific Chunk

In [ ]:
chunk = chunks.items[1]  # Get the AI-related chunk
retrieved_chunk = client.get_chunk(chunk.id)
retrieved_chunk.text

## Update a Chunk's Embedding

In [ ]:
# Update the chunk with a completely different text and embedding
new_text = "Dogs are loyal pets and great companions"
new_embedding = co.embed(
    texts=[new_text],
    model="embed-english-v3.0",
    input_type="search_document"
).embeddings[0]

updated_chunk = client.update_chunk(
    chunk_id=chunk.id,
    text=new_text,
    embedding=new_embedding
)

## Search Again - See How Results Changed

In [ ]:
# Same AI query as before
results = client.search_library(
    library_id=library.id,
    query=query_embedding,
    top_k=3
)

print("Results after update:")
for result in results.results:
    print(f"Score: {result.score:.4f}")
    print(f"Text: {result.chunk.text}")
    print()

## Delete a Chunk

In [ ]:
# Delete the NLP-related chunk
nlp_chunk = chunks.items[4]
client.delete_chunk(nlp_chunk.id)

## Search After Deletion

In [ ]:
# Search for text processing
text_query = "How do computers process language?"
text_query_embedding = co.embed(
    texts=[text_query],
    model="embed-english-v3.0",
    input_type="search_query"
).embeddings[0]

results = client.search_library(
    library_id=library.id,
    query=text_query_embedding,
    top_k=5
)

print(f"Found {len(results.results)} results (NLP chunk was deleted):")
for result in results.results:
    print(f"- {result.chunk.text}")

## Cleanup

In [ ]:
# Uncomment to delete
client.delete_library(library.id)